# Retail Inventory Dataset Cleaning

Clean, standardize, validate, and save the messy retail inventory dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Retail_Messy_46.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (46, 6)


## Standardize Text and Missing Values

In [3]:
text_cols = ["SKU", "ProductName", "StockStatus"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in text_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
SKU             0
ProductName     0
StockQty        0
UnitCost        0
ReorderLevel    3
StockStatus     0
dtype: int64


## Clean Numeric Inventory Fields

In [ ]:
numeric_cols = ["StockQty", "UnitCost", "ReorderLevel"]
raw_values = df["StockQty"].astype("string")
cleaned_values = raw_values.str.replace(r"[^0-9.\-]", "", regex=True)
df["StockQty"] = pd.to_numeric(cleaned_values, errors="coerce")
df.loc[(df["StockQty"] < 0) | (df["StockQty"] > 1000), "StockQty"] = np.nan
df["StockQty"] = df["StockQty"].fillna(df["StockQty"].median()).round().astype("int64")

print("StockQty summary:")
print(df["StockQty"].describe())

Invalid or corrected values: {'StockQty': 0, 'UnitCost': 0, 'ReorderLevel': 0}
         StockQty     UnitCost  ReorderLevel
count   46.000000    46.000000     46.000000
mean    42.521739  1504.434783     19.782609
std     30.829812  1442.344860      5.159298
min      0.000000   149.000000     10.000000
25%     13.500000   499.000000     15.000000
50%     40.000000   899.000000     20.000000
75%     60.000000  2199.000000     25.000000
max    120.000000  4499.000000     25.000000


In [ ]:
raw_values = df["UnitCost"].astype("string")
cleaned_values = raw_values.str.replace(r"[^0-9.\-]", "", regex=True)
df["UnitCost"] = pd.to_numeric(cleaned_values, errors="coerce")
df.loc[df["UnitCost"] <= 0, "UnitCost"] = np.nan
df["UnitCost"] = df["UnitCost"].fillna(df["UnitCost"].median()).round().astype("int64")

print("UnitCost summary:")
print(df["UnitCost"].describe())

In [ ]:
raw_values = df["ReorderLevel"].astype("string")
cleaned_values = raw_values.str.replace(r"[^0-9.\-]", "", regex=True)
df["ReorderLevel"] = pd.to_numeric(cleaned_values, errors="coerce")
df.loc[df["ReorderLevel"] < 0, "ReorderLevel"] = np.nan
df["ReorderLevel"] = df["ReorderLevel"].fillna(df["ReorderLevel"].median()).round().astype("int64")

print("ReorderLevel summary:")
print(df["ReorderLevel"].describe())

## Normalize Stock Status and Remove Duplicates

In [6]:
df["StockStatus"] = (
    df["StockStatus"]
    .str.lower()
    .map({
        "in stock": "In Stock",
        "low stock": "Low Stock",
        "out of stock": "Out of Stock",
    })
)

df["StockStatus"] = np.select(
    [df["StockQty"] == 0, df["StockQty"] <= df["ReorderLevel"]],
    ["Out of Stock", "Low Stock"],
    default="In Stock",
)

duplicate_count = int(df["SKU"].duplicated().sum())
df = df.drop_duplicates(subset="SKU", keep="first").reset_index(drop=True)

print("Duplicates removed:", duplicate_count)
print(df["StockStatus"].value_counts())

Duplicates removed: 1
StockStatus
In Stock        32
Low Stock       10
Out of Stock     3
Name: count, dtype: int64


## Validate the Cleaned Dataset

In [7]:
expected_statuses = {"In Stock", "Low Stock", "Out of Stock"}

assert df["SKU"].notna().all()
assert df["SKU"].is_unique
assert df["ProductName"].notna().all()
assert df[numeric_cols].notna().all().all()
assert (df["StockQty"] >= 0).all()
assert (df["UnitCost"] > 0).all()
assert (df["ReorderLevel"] >= 0).all()
assert set(df["StockStatus"]) <= expected_statuses
assert (
    df["StockStatus"]
    == np.select(
        [df["StockQty"] == 0, df["StockQty"] <= df["ReorderLevel"]],
        ["Out of Stock", "Low Stock"],
        default="In Stock",
    )
 ).all()

print("Validation passed")
print("Final shape:", df.shape)
print("Missing values:")
print(df.isna().sum())

Validation passed
Final shape: (45, 6)
Missing values:
SKU             0
ProductName     0
StockQty        0
UnitCost        0
ReorderLevel    0
StockStatus     0
dtype: int64


## Save the Cleaned Dataset

In [ ]:
output_path = "Retail_Cleaned_46.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Inventory\Retail_Cleaned_46.csv
Saved shape: (45, 6)
